In [1]:
# Milestone Four Enhancements
# Dash/Jupyter
from jupyter_dash import JupyterDash

# Dash components
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
import dash_leaflet as dl

# Data/Viz
import pandas as pd
import plotly.express as px

import base64, os

# CRUD module
from animal_shelter import AnimalShelter

# -------------------------------
# DB CONNECTION
# -------------------------------
try:
    db = AnimalShelter("aacuser", "SNHU1234")
except Exception as e:
    print(f"Database connection failed: {e}")
    db = None

# Load initial data
if db:
    df = pd.DataFrame.from_records(db.read({}, {"_id": False}))
else:
    df = pd.DataFrame()

df = df.drop(columns=["_id"], errors="ignore")

# -------------------------------
# QUERY DEFINITIONS
# -------------------------------
QUERY_MAP = {
    "Water": {
        "animal_type": "Dog",
        "breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]},
        "sex_upon_outcome": "Intact Female",
        "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
    },
    "Mountain": {
        "animal_type": "Dog",
        "breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]},
        "sex_upon_outcome": "Intact Male",
        "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
    },
    "Disaster": {
        "animal_type": "Dog",
        "breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"]},
        "sex_upon_outcome": "Intact Male",
        "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
    }
}

# -------------------------------
# DATA CLEANING
# -------------------------------
def clean_data(df):
    if df.empty:
        return df
    df = df.dropna(subset=["breed", "animal_type"])
    df["breed"] = df["breed"].str.strip()
    return df

# -------------------------------
# DYNAMIC QUERY BUILDER (NEW)
# -------------------------------
def build_query(filter_value, search_value):
    query = {}

    # Apply preset filters
    if filter_value in QUERY_MAP:
        query.update(QUERY_MAP[filter_value])

    # Apply search using MongoDB regex
    if search_value:
        query["$or"] = [
            {"breed": {"$regex": search_value, "$options": "i"}},
            {"animal_type": {"$regex": search_value, "$options": "i"}}
        ]

    return query

# -------------------------------
# SORTING FUNCTION
# -------------------------------
def get_top_animals_by_age(df, top_n=10):
    if df.empty or "age_upon_outcome_in_weeks" not in df.columns:
        return df
    return df.sort_values(by="age_upon_outcome_in_weeks", ascending=False).head(top_n)

# -------------------------------
# AGGREGATION FUNCTION
# -------------------------------
def breed_statistics(df):
    if df.empty or "breed" not in df.columns:
        return pd.DataFrame()

    stats = df.groupby("breed").size().reset_index(name="count")
    return stats.sort_values(by="count", ascending=False).head(10)

# -------------------------------
# DASH APP
# -------------------------------
app = JupyterDash(__name__)

logo_path = "Grazioso Salvare Logo.png"
logo_b64 = base64.b64encode(open(logo_path, "rb").read()).decode("utf-8") if os.path.exists(logo_path) else None

UNIQUE_ID = "Amanda Celi, CS-340"

app.layout = html.Div([

    html.Div([
        html.Center(html.B(html.H1("CS-340 Dashboard"))),
        html.Center(html.Div(UNIQUE_ID, style={"fontWeight": "bold"})),
        html.Center(html.Img(src=f"data:image/png;base64,{logo_b64}", style={"height": "48px"})) if logo_b64 else html.Span(),
    ]),
    html.Hr(),

    # SEARCH BAR
    html.Div([
        html.Label("Search Animals"),
        dcc.Input(id="search-box", type="text", placeholder="Enter keyword...", debounce=True),
    ], style={"marginBottom": "10px"}),

    # FILTER OPTIONS
    html.Div([
        html.Label("Filter Options"),
        dcc.RadioItems(
            id="filter-type",
            options=[
                {"label": "Water Rescue", "value": "Water"},
                {"label": "Mountain Rescue", "value": "Mountain"},
                {"label": "Disaster Rescue", "value": "Disaster"},
                {"label": "Reset", "value": "Reset"}
            ],
            value="Reset",
            inline=True
        ),
        html.Div(id="records-count")
    ]),
    html.Hr(),

    dash_table.DataTable(
        id="datatable-id",
        columns=[{"name": c, "id": c} for c in df.columns],
        data=df.to_dict("records"),
        filter_action="native",
        sort_action="native",
        page_size=10,
        row_selectable="single",
        selected_rows=[0],
    ),

    html.Br(),
    html.Div(style={"display": "flex"}, children=[
        html.Div(id="graph-id", style={"width": "50%"}),
        html.Div(id="map-id", style={"width": "50%"})
    ])
])

# -------------------------------
# CALLBACK: TABLE UPDATE
# -------------------------------
@app.callback(
    Output("datatable-id", "data"),
    Output("records-count", "children"),
    Input("filter-type", "value"),
    Input("search-box", "value")
)
def update_table(filter_value, search_value):

    if not db:
        return [], "Database not available"

    # 🔥 DATABASE-LEVEL QUERY
    query = build_query(filter_value, search_value)

    recs = db.read(query, {"_id": False})
    dff = pd.DataFrame.from_records(recs)

    # Clean data
    dff = clean_data(dff)

    # Sort
    dff = get_top_animals_by_age(dff)

    message = f"{len(dff)} records shown"
    if len(dff) == 0:
        message = "No results found"

    return dff.to_dict("records"), message

# -------------------------------
# CALLBACK: CHART
# -------------------------------
@app.callback(
    Output("graph-id", "children"),
    Input("datatable-id", "data")
)
def update_chart(table_data):

    dff = pd.DataFrame(table_data)

    stats = breed_statistics(dff)

    if stats.empty:
        return html.Div("No data to chart")

    fig = px.pie(stats, values="count", names="breed", title="Breed Distribution")
    return dcc.Graph(figure=fig)

# -------------------------------
# CALLBACK: MAP
# -------------------------------
@app.callback(
    Output("map-id", "children"),
    Input("datatable-id", "data"),
    Input("datatable-id", "selected_rows")
)
def update_map(table_data, selected_rows):

    def base_map():
        return dl.Map(center=[30.75, -97.48], zoom=10, children=[dl.TileLayer()])

    if not table_data:
        return base_map()

    dff = pd.DataFrame(table_data)

    row = selected_rows[0] if selected_rows else 0
    record = dff.iloc[row]

    try:
        lat = float(record["location_lat"])
        lon = float(record["location_long"])
    except:
        return base_map()

    return dl.Map(center=[lat, lon], zoom=12, children=[
        dl.TileLayer(),
        dl.Marker(position=[lat, lon])
    ])

# -------------------------------
# RUN APP
# -------------------------------
app.run(debug=True)

c:\Users\amand\AppData\Local\Programs\Python\Python313\Lib\site-packages\dash\dash.py:644: UserWarning: JupyterDash is deprecated, use Dash instead.
See https://dash.plotly.com/dash-in-jupyter for more details.
  warnings.warn(
